In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import shap
import matplotlib.pyplot as plt

2026-02-11 23:04:16.295065: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-11 23:04:16.315793: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-11 23:04:17.104976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-11 23:04:19.556839: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [2]:
CURRENT_DIR = Path.cwd()
checkpoint_path = CURRENT_DIR.parent / "model_training" / "checkpoints" / "final_model.h5"
sys.path.append(str(CURRENT_DIR.parent / "data_preprocessing"))
from normalize_fn import load
print("Checkpoint path:", checkpoint_path.resolve())

Checkpoint path: /mnt/c/Users/mayak/DeepSyn/src/model_training/checkpoints/final_model.h5


In [3]:
norm='norm'
X_tr, X_val, _, _, y_tr, y_val, _, _=load(norm=norm)
if not isinstance(X_val, pd.DataFrame):
    X_val = pd.DataFrame(X_val, columns=[f"Feature_{i}" for i in range(X_val.shape[1])])
print("Validation data shape:", X_val.shape)

Validation data shape: (4614, 7063)


In [4]:
if checkpoint_path.exists():
    print("Loading trained model...")
    model=tf.keras.models.load_model(str(checkpoint_path))
else:
    raise FileNotFoundError(f"Checkpoint not found at {checkpoint_path.resolve()}")

Loading trained model...


/home/mayak/dsenv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-02-11 23:06:11.973047: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [5]:
background = X_val.sample(n=min(100, len(X_val)), random_state=42).values

In [6]:
explainer = shap.DeepExplainer(model, background)
shap_values = explainer.shap_values(X_val.values)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.array(shap_values)
if len(shap_values.shape) == 3:  
    shap_values = shap_values[:, 0, :]
print("SHAP values shape:", shap_values.shape)

/home/mayak/dsenv/lib/python3.10/site-packages/shap/explainers/_deep/deep_tf.py:94: UserWarning: Your TensorFlow version is newer than 2.4.0 and so graph support has been removed in eager mode and some static graphs may not be supported. See PR #1483 for discussion.
  warnings.warn(
/home/mayak/dsenv/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: dense_input
Received: inputs=['Tensor(shape=(100, 7063))']
  warnings.warn(msg)
/home/mayak/dsenv/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: dense_input
Received: inputs=['Tensor(shape=(200, 7063))']
  warnings.warn(msg)
/home/mayak/dsenv/lib/python3.10/site-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: dense_input
Received: inputs=['Tensor(shape=(4

SHAP values shape: (4614, 1)


In [ ]:
mean_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'Feature': X_val.columns,
    'Importance': mean_shap}).sort_values(by='Importance', ascending=False)
top5_features = feature_importance.head(5)
print("Global Top 5 Features:")
print(top5_features)

In [ ]:
plt.figure(figsize=(8,5))
plt.barh(top5_features['Feature'][::-1],top5_features['Importance'][::-1], color='skyblue')
plt.xlabel("Mean SHAP value")
plt.title("Top 5 Global Features by SHAP")
plt.tight_layout()
plt.show()